# Обучение модели регрессии для CC50

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.svm import SVR

from scipy.stats import loguniform

from catboost import CatBoostRegressor

import tqdm
from tqdm.auto import tqdm

import pickle

In [2]:
RANDOM_STATE = 22

Загружаем данные. Признаки уже стандартизированы, а таргет логарифмирован.

In [3]:
train_df = pd.read_csv("../../data/processed/cc50_train.csv")
test_df = pd.read_csv("../../data/processed/cc50_test.csv")

In [4]:
X_train = train_df.drop(columns=["CC50, mM"])
y_train = train_df["CC50, mM"]

X_test = test_df.drop(columns=["CC50, mM"])
y_test = test_df["CC50, mM"]

print(f"В обучающей выборке содержится {X_train.shape[0]} строк и {X_train.shape[1]} признаков")
print(f"В тестовой выборке содержится {X_test.shape[0]} строк и {X_test.shape[1]} признаков")

В обучающей выборке содержится 798 строк и 30 признаков
В тестовой выборке содержится 200 строк и 30 признаков


Зададим параметры сетки для подбора параметров четырех разных моделей регрессии.

In [5]:
param_distributions = {
    "RidgeRegression": {
        'model': Ridge(),
        'params': {
            'alpha': loguniform(1e-2, 1e3)
        }
    },
    "RandomForest": {
        'model': RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1),
        'params': {
            'n_estimators': [100, 150, 200],
            'max_depth': [3, 5, 8, 12],
            'min_samples_split': [2, 5, 10],
            'min_samples_leaf': [1, 2, 4]
        }
    },
    "CatBoost": {
        "model": CatBoostRegressor(
            allow_writing_files=False,
            loss_function="RMSE", 
            verbose=False, 
            thread_count=1, 
            random_seed=RANDOM_STATE, 
            boosting_type="Plain",
            grow_policy="Lossguide",
            leaf_estimation_iterations=1
        ),
        "params": {
            "iterations": [150, 300, 500],
            "learning_rate": np.logspace(-2, -1, 5),
            "depth": [4, 5, 6, 7],
            "l2_leaf_reg": [3, 5, 10, 15],
            "max_leaves": [30, 50],
        }
    },
    "SVR": {
        "model": SVR(kernel="rbf"),
        "params": {
            "C": np.logspace(-2, 3, 20),  # От 0.01 до 1000
            "gamma": np.logspace(-4, 1, 20),  # От 0.0001 до 10
            "epsilon": [0.01, 0.05, 0.1, 0.2],  # Чувствительность к ошибкам
        }
    }
}

In [6]:
best_models = {}
for name, config in tqdm(param_distributions.items()):
    print(f"\n  Модель: {name}")
    print("="*50)
    
    grid_search = RandomizedSearchCV(
        random_state=RANDOM_STATE,
        estimator=config['model'],
        param_distributions=config['params'],
        cv=5,
        scoring='r2',
        n_iter=20,
        n_jobs=-1
    )
    
    grid_search.fit(X_train, y_train)
    
    # Оцениваем лучшую модель на тестовой выборке
    best_model = grid_search.best_estimator_
    best_models[name] = best_model
    
    preds = best_model.predict(X_test)
    r2 = r2_score(y_test, preds)

    # MAE оценим на данных в реальном масштабе
    mae = mean_absolute_error(10**(-y_test), 10**(-preds))
    print(f"    Лучшие параметры: {grid_search.best_params_}")
    print(f"    Тестовый R²: {r2:.4f}")
    print(f"    Тестовый MAE: {mae:.4f}")

  0%|          | 0/4 [00:00<?, ?it/s]


  Модель: RidgeRegression
    Лучшие параметры: {'alpha': np.float64(615.4047449570229)}
    Тестовый R²: 0.1640
    Тестовый MAE: 471.7272

  Модель: RandomForest
    Лучшие параметры: {'n_estimators': 200, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_depth': 12}
    Тестовый R²: 0.3659
    Тестовый MAE: 327.7947

  Модель: CatBoost
    Лучшие параметры: {'max_leaves': 30, 'learning_rate': np.float64(0.01778279410038923), 'l2_leaf_reg': 3, 'iterations': 500, 'depth': 5}
    Тестовый R²: 0.3619
    Тестовый MAE: 331.2221

  Модель: SVR
    Лучшие параметры: {'gamma': np.float64(0.023357214690901212), 'epsilon': 0.2, 'C': np.float64(4.281332398719392)}
    Тестовый R²: 0.4331
    Тестовый MAE: 355.5555


Лучшей моделью по петрике $R^2$ показал себя метод опорных векторов SVR с параметрами 

{'gamma': np.float64(0.023357214690901212),<br> 'epsilon': 0.2,<br> 'C': np.float64(4.281332398719392)}.

Сериализуем лучшую модель и сохраним в файл.

In [7]:
with open('../../models/model_cc50_regression.pkl', 'wb') as output:
    pickle.dump(best_models["SVR"], output)